### Encoder 기반 : 주어진 텍스트를 깊이있게 읽고 이해 - BERT
### Decoder 기반 : 텍스트를 창작하고 써내려감 - GPT

### BERT VS GPT 태생이 다름.. 근본차이는 Attention이 문장을 어떻게 처리(보는가)하는데 있음
- BERT는 양방향 : 문장 전체를 한번에 통째로 본다( 중간단어 맞추는 학습)  독해 분류에 최적화
- GPT 단방향 : 오직 과거와 현재만 본다 " 나는 밥을 " 여기까지보면 "먹는다"를 예측
    - 뒤를 미리 볼수없기때문에 미래로 가버리는 특수한 장치가 있음
    - Masked Self-Attention ( 치팅 방지 기법)
        - 문장을 훈련할때 문장전체를 통째로 주면 다음단어를 예측하지 않고 그냥 옆에 있는 정답을 컨닝(Cheating)
        - 행렬 연산을 할때 마스킹 처리해서 softmax함수를 통과하면 확률이 0이되게 무시
    - 자기회귀(Autoregresisive)        
        - 사용자가 "옛날 옛적에" 라고 입력하면 gpt 다음 '한' 이라고 예측하면
        - "옛날 옛적에 한" 을 만들고 다시 다음 단어 예측... 반복
        - 끝을 나타내는 토큰 [EOF] 나올때까지 무한반복
        -  디코딩전략(Decoding Strategy) : 다음 단어를 선택할때  확률이 높은단어? 아니면 가뜸 랜덤하게 엉뚱한 단어를 선택?

In [10]:
import torch
from transformers import GPT2LMHeadModel
from transformers import PreTrainedTokenizerFast

tokenizer = PreTrainedTokenizerFast.from_pretrained("skt/kogpt2-base-v2",
bos_token='</s>', eos_token='</s>', unk_token='<unk>',
pad_token='<pad>', mask_token='<mask>')

model = GPT2LMHeadModel.from_pretrained('skt/kogpt2-base-v2')

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

[transformers] GPT2LMHeadModel LOAD REPORT from: skt/kogpt2-base-v2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [11]:
# 앵무새병... 1등 단어만 맹목적으로 고르는 Greedy 특성상 한번 특정문구패턴이 1등 확률을차지하지 시작하면 반복됨
prompt_text = "인공지능이 인간을 대처할 수 있을까?"
input_ids = tokenizer.encode(prompt_text,return_tensors='pt')

with torch.no_grad():
    greedy_output =  model.generate(
        input_ids,
        max_length = 50,
        pad_token_id = tokenizer.pad_token_id
    )
    
generated_text = tokenizer.decode(greedy_output[0],skip_special_tokens=True)    
print(generated_text)

인공지능이 인간을 대처할 수 있을까?"라며 "그런데도 우리는 인공지능이 인간을 어떻게 대처할 수 있는지 알지 못한다"고 말했다.
그는 "인공지능은 인간의 행동을 예측하고 통제할 수 있는 능력을 갖고


In [12]:
# Sampling : 무조건 1등 단어만 뽑는게 아니라 주사위 굴리듯이 확률에 기반하여 가끔은 2등 3등 단어도 뽑아주는 랜덤성 기법 도입
with torch.no_grad():
    sample_output =  model.generate(
        input_ids,
        max_length = 50,
        do_sample=True,  # 주사위 굴리기
        top_k = 50,  # 상위50개중에서 주사위 굴리기
        top_p = 0.9, # 누적확률 90% 안에서
        temperature=0.8, # 1이면 미친 창의성  0.1 , 0 완전 안전하게 확률에 기반(창작을 거의 안함)
        repetition_penalty = 1.2,  # 했던 말 또하면 벌점
        pad_token_id = tokenizer.pad_token_id
    )
creative_text = tokenizer.decode(sample_output[0], skip_special_tokens=True)    
print(creative_text)

인공지능이 인간을 대처할 수 있을까?"라며 "그렇다면 인간의 두뇌가 얼마나 중요한지를 보여주는 증거"라고 말했다.
또한 "과학적 지식이 미래 사회의 가장 큰 화두가 될 것 같다"면서 "이런 상황에서 인간은 어떤 생각을 하는


### gpt-2 파인튜닝
- GPT는 단순히 아무말이나 내 뱉고잇음.. 이를 통제-> 파인튜닝 Q&A
- BERT 계열은 labels에 정답을 제공했지만 GPT는 문장생성->내가 제공한 문장 다음에 올 단어를 예측
- 허깅페이스 라이브러리는 lbels에 input_ids를 전체를 주면 내부적으로 정답지를 한칸씩 오른쪽으로 밀어서 만들어줌

In [17]:
from torch.optim import AdamW
train_text = [
    '질문 : 자연어 처리가 뭔가요? 답변 : 컴퓨터가 인간의 언어를 이해하고 분석하는 인공지능 기술입니다.',
    '질문 : 오늘 기분이 어때? 답변 : 저는 인공지능이라 기분을 느낄수 없지만, 당신과 대화해서 기뻐요'
]
inputs = tokenizer(train_text,padding=True, truncation=True, return_tensors='pt')
input_ids = inputs['input_ids']
attention_mask = inputs['attention_mask']
optimizer = AdamW(model.parameters(), lr=5e-5)
model.train()
epochs = 10
for epoch in range(epochs):
    optimizer.zero_grad()
    outputs = model(input_ids=input_ids,attention_mask=attention_mask,labels=input_ids)
    loss = outputs.loss
    loss.backward()
    optimizer.step()
    print(f'epoch {epoch+1}/{epochs} loss = {loss.item()}')
model.eval()
test_prompt = '질문:오늘 기분은? 답변:'    
test_ids = tokenizer.encode(test_prompt,return_tensors='pt')
with torch.no_grad():
    test_output = model.generate(
        test_ids,
        max_length = 40,
        pad_token_id = tokenizer.pad_token_id
    )
print( tokenizer.decode(test_output[0],skip_special_tokens=True)     )

epoch 1/10 loss = 0.03429316729307175
epoch 2/10 loss = 0.5186513662338257
epoch 3/10 loss = 0.24373020231723785
epoch 4/10 loss = 0.0656362920999527
epoch 5/10 loss = 0.04260675236582756
epoch 6/10 loss = 0.10883012413978577
epoch 7/10 loss = 0.10805174708366394
epoch 8/10 loss = 0.07379725575447083
epoch 9/10 loss = 0.05342349037528038
epoch 10/10 loss = 0.04443536698818207
질문:오늘 기분은? 답변: 저는 인공지능이라 기분을 느낄수 없지만, 당신과 대화해서 기뻐요? 당신과 대화해서 기뻐요? 당신과 대화해서 기뻐요


In [ ]:
model.eval()
test_prompt = '질문:철수가 영희를 괴롭히면 어떻게 할까요? 답변:'
test_ids = tokenizer.encode(test_prompt,return_tensors='pt')
with torch.no_grad():
    test_output = model.generate(
        test_ids,
        max_length = 40,
        temperature=0.8,
        pad_token_id = tokenizer.pad_token_id
    )
print( tokenizer.decode(test_output[0],skip_special_tokens=True)     )

질문:철수가 영희를 괴롭히면 어떻게 할까요? 답변: 저는 인공지능이라 기분을 느낄수 없지만, 당신과 대화해서 기뻐요? 당신과 대화해서 기뻐
